# Experto 4 — LUNA16: Swin3D (inflación timm) + Grad-CAM

Notebook de entrenamiento del **Experto 4** del sistema MoE.

- **Origen de datos:** el zip **`Luna16 Lung Cancer Dataset.zip`** en `01_Data/Raw/` (misma convención que `01_Exploracion_Estructura_Datasets.ipynb`). Tras copiarlo a `/content/datasets/` se descomprime en **`/content/datasets/Luna16 Lung Cancer Dataset/`**. Ahí están los **888** `.mhd`, el binario (`.raw`/`.zraw`) y los **CSV del challenge** (`annotations.csv`, `candidates_V2.csv`, etc.); **no se genera ningún CSV nuevo**: solo se leen con `pandas`.
- **Volumen ROI:** cubo **64×64×64** vóxeles tras resample 1 mm y ventana HU (alineado al paper y a la consigna MoE).
- **Etiquetas `class`:** se toman del CSV de candidatos del zip (track reducción de FP: 0/1). No es malignidad LIDC; si más adelante usas `processed_labels.csv` con malignancy, sustituye esa lectura.
- **Arquitectura:** Swin-Tiny (timm) con patch_embed inflado a Conv3d; plan B R3D-18.
- **Referencia:** `guides/Luna16/Deep learning-based lung cancer classification of CT images.md`


In [ ]:
# Fase 0: Setup & montar Drive
!pip install -q SimpleITK nibabel monai timm scikit-learn tqdm seaborn scipy

from google.colab import drive
import os, glob, time, json, shutil, subprocess, random
import numpy as np
import pandas as pd
import SimpleITK as sitk
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler
from torch.utils.checkpoint import checkpoint as grad_checkpoint
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix,
                             classification_report)
from tqdm.auto import tqdm
import timm
import matplotlib.pyplot as plt

drive.mount('/content/drive')

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")

# Misma convención que 01_Exploracion_Estructura_Datasets.ipynb
RAW_DIR = '/content/drive/MyDrive/PROYECTO_MOE_VISION/01_Data/Raw/'
LOCAL_DEST = '/content/datasets/'
LUNA_ZIP_NAME = 'Luna16 Lung Cancer Dataset.zip'
LUNA_FOLDER_NAME = os.path.splitext(LUNA_ZIP_NAME)[0]  # "Luna16 Lung Cancer Dataset"
LUNA_BASE = os.path.join(LOCAL_DEST, LUNA_FOLDER_NAME)

WEIGHTS_DIR = '/content/drive/MyDrive/PROYECTO_MOE_VISION/03_Weights/Expert4_LUNA16_Swin3D/'
for d in ['checkpoints', 'metrics', 'figures']:
    os.makedirs(os.path.join(WEIGHTS_DIR, d), exist_ok=True)
os.makedirs(LOCAL_DEST, exist_ok=True)
print("Rutas configuradas.")
print(f"  Zip en Drive: {os.path.join(RAW_DIR, LUNA_ZIP_NAME)}")
print(f"  Datos locales (CSV + .mhd vienen dentro del zip): {LUNA_BASE}")


In [ ]:
# Fase 1: LUNA16 — mismo flujo que 01_Exploracion_Estructura_Datasets.ipynb
# 1) Copiar el zip desde Drive a /content/datasets/
# 2) Descomprimir en /content/datasets/Luna16 Lung Cancer Dataset/
# 3) Borrar el zip local (libera espacio en Colab)
# Los CSV (annotations, candidates_V2, etc.) ya vienen dentro del zip; no se crean a mano.

FORCE_REUNZIP = False  # True para forzar descompresión aunque existan .mhd

drive_zip_path = os.path.join(RAW_DIR, LUNA_ZIP_NAME)
if not os.path.isfile(drive_zip_path):
    alt = sorted(
        glob.glob(os.path.join(RAW_DIR, '*Luna*.zip')) +
        glob.glob(os.path.join(RAW_DIR, '*luna*.zip'))
    )
    if alt:
        drive_zip_path = alt[0]
        print(f"AVISO: usando zip encontrado: {drive_zip_path}")
    else:
        raise FileNotFoundError(
            f"No está {LUNA_ZIP_NAME} en {RAW_DIR}. Sube el zip o ajusta LUNA_ZIP_NAME."
        )

local_zip_path = os.path.join(LOCAL_DEST, os.path.basename(drive_zip_path))
unzip_dir = os.path.join(LOCAL_DEST, os.path.splitext(os.path.basename(drive_zip_path))[0])

mhd_probe = glob.glob(os.path.join(unzip_dir, '**', '*.mhd'), recursive=True)
if FORCE_REUNZIP or len(mhd_probe) < 100:
    print("1. Copiando zip desde Drive al entorno local...")
    t0 = time.time()
    shutil.copy2(drive_zip_path, local_zip_path)
    print(f"   Copia OK ({time.time()-t0:.1f}s)")

    os.makedirs(unzip_dir, exist_ok=True)
    print(f"2. Descomprimiendo en {unzip_dir} ...")
    t1 = time.time()
    r = subprocess.run(
        ['unzip', '-q', '-o', local_zip_path, '-d', unzip_dir],
        capture_output=True, text=True,
    )
    if r.returncode != 0:
        print("unzip falló, intentando zipfile...")
        import zipfile
        with zipfile.ZipFile(local_zip_path, 'r') as zf:
            zf.extractall(unzip_dir)
    print(f"   Unzip OK ({time.time()-t1:.1f}s)")

    print("3. Borrando zip local...")
    os.remove(local_zip_path)
else:
    print(f"Datos ya presentes en {unzip_dir} ({len(mhd_probe)} .mhd). "
          f"Pon FORCE_REUNZIP=True para volver a extraer.")

LUNA_BASE = unzip_dir  # raíz del dataset descomprimido (incluye CSV del challenge)

mhd_files = sorted(glob.glob(os.path.join(LUNA_BASE, '**', '*.mhd'), recursive=True))
raw_files = sorted(
    glob.glob(os.path.join(LUNA_BASE, '**', '*.raw'), recursive=True)
    + glob.glob(os.path.join(LUNA_BASE, '**', '*.zraw'), recursive=True)
)
csv_files = sorted(glob.glob(os.path.join(LUNA_BASE, '**', '*.csv'), recursive=True))

print(f"\nArchivos encontrados bajo {LUNA_BASE}:")
print(f"  .mhd: {len(mhd_files)}")
print(f"  .raw/.zraw: {len(raw_files)}")
print(f"  .csv: {len(csv_files)}")
for c in csv_files:
    print(f"    {c}")


In [ ]:
# --- Descubrimiento: ¿dónde está la etiqueta del dataset? (CSV vs .mhd) ---
# Ejecutar después de la Fase 1 (tener LUNA_BASE, mhd_files, csv_files definidos).

print("=" * 70)
print("ETIQUETAS LUNA16: ¿en el CSV o en el volumen .mhd?")
print("=" * 70)

print("""
Resumen esperado (challenge LUNA16):
- Los archivos .mhd (+ .raw/.zraw) son SOLO imágenes CT (valores HU, geometría).
  No incluyen la etiqueta de clase del track de candidatos ni malignidad clínica.
- Las columnas de coordenadas y la columna `class` (0/1) vienen en los CSV del zip,
  en particular candidates / candidates_V2 para el track de falsos positivos.
- annotations.csv da posición y tamaño del nódulo de referencia (detección), no
  benigno/maligno al estilo LIDC.
""")

print("\n[1] CSV encontrados en el dataset descomprimido:\n")
for cpath in csv_files:
    bn = os.path.basename(cpath)
    try:
        df = pd.read_csv(cpath, nrows=5)
        cols = list(df.columns)
        print(f"  • {bn}")
        print(f"    columnas ({len(cols)}): {cols}")
        if "class" in [x.lower() for x in cols]:
            print("    → contiene columna 'class' (etiqueta 0/1 por candidato).")
        if "malign" in " ".join(cols).lower():
            print("    → contiene algo relacionado con malignidad (revisar).")
    except Exception as e:
        print(f"  • {bn}  ERROR leyendo: {e}")

print("\n[2] Un archivo .mhd de ejemplo (metadatos SimpleITK):\n")
if mhd_files:
    sample_mhd = mhd_files[0]
    print(f"  Archivo: {sample_mhd}")
    img = sitk.ReadImage(sample_mhd)
    keys = list(img.GetMetaDataKeys())
    print(f"  Número de claves de metadatos: {len(keys)}")
    for k in keys[:25]:
        try:
            v = img.GetMetaData(k)
            if len(v) > 80:
                v = v[:77] + "..."
            print(f"    {k}: {v}")
        except Exception:
            print(f"    {k}: <no legible>")
    if len(keys) > 25:
        print(f"    ... y {len(keys)-25} claves más.")
    print("""
  Interpretación: suele haber DICOM/Series UID, espaciado, etc. No esperes una
  columna 'label' o 'cancer' aquí salvo que tu pipeline haya escrito algo
  personalizado (p. ej. otro notebook que añadió RouterLabel al convertir a .mha).
""")
else:
    print("  (No hay .mhd todavía — ejecuta la celda de descompresión primero.)")

print("\n[3] Conclusión para este notebook:\n")
print("  → La etiqueta que usa el DataLoader (columna `label`) se arma leyendo")
print("    el CSV de candidatos con `class`, NO leyendo el .mhd.")
print("  → Si necesitas benigno/maligno como en papers con LIDC, hay que unir")
print("    con tablas de malignancy externas o un CSV procesado aparte.\n")
print("=" * 70)


In [ ]:
# Fase 2: Leer CSVs que ya vienen en el zip (no generamos ningún CSV en disco)
# - annotations.csv: referencia de detección (coords mundo, diámetro)
# - candidates_V2.csv: track FP reduction (class 0 = no-nódulo, 1 = nódulo)
# Malignidad binaria estilo paper DCSwinB requiere tablas LIDC aparte; aquí usamos
# las etiquetas del challenge oficial en candidates_V2 cuando exista.

def _normalize_luna_columns(df):
    df = df.copy()
    df.columns = [str(c).strip() for c in df.columns]
    lower = {c.lower(): c for c in df.columns}
    ren = {}
    if 'seriesuid' not in lower and 'seriesinstanceuid' in lower:
        ren[lower['seriesinstanceuid']] = 'seriesuid'
    for a, b in [('coordx', 'coordX'), ('coordy', 'coordY'), ('coordz', 'coordZ')]:
        if a in lower and b not in lower:
            ren[lower[a]] = b
    for old, new in ren.items():
        df = df.rename(columns={old: new})
    return df

ann_path = os.path.join(LUNA_BASE, 'annotations.csv')
if not os.path.isfile(ann_path):
    ann_path = next((c for c in csv_files if 'annotations' in os.path.basename(c).lower()
                     and 'excluded' not in os.path.basename(c).lower()), None)

cand_path = os.path.join(LUNA_BASE, 'candidates_V2.csv')
if not os.path.isfile(cand_path):
    cand_path = next((c for c in csv_files if 'candidates_v2' in os.path.basename(c).lower()), None)
if cand_path is None or not os.path.isfile(str(cand_path)):
    cand_path = next((c for c in csv_files if 'candidates' in os.path.basename(c).lower()), None)

print(f"annotations.csv: {ann_path}")
print(f"candidates CSV:  {cand_path}")

annotations = pd.read_csv(ann_path) if ann_path and os.path.isfile(str(ann_path)) else pd.DataFrame()
if len(annotations) > 0:
    annotations = _normalize_luna_columns(annotations)
print(f"Filas en annotations (referencia detección): {len(annotations)}")
if len(annotations) > 0:
    print(annotations.head())

uid_to_mhd = {}
for mhd in mhd_files:
    uid = os.path.splitext(os.path.basename(mhd))[0]
    uid_to_mhd[uid] = mhd

if cand_path is not None and os.path.isfile(str(cand_path)):
    candidates = pd.read_csv(cand_path)
    candidates = _normalize_luna_columns(candidates)
    candidates['class'] = pd.to_numeric(candidates['class'], errors='coerce').fillna(0).astype(int)
    print(f"\nCandidates (desde zip) shape: {candidates.shape}")
    print(candidates.head())
    nodules = candidates[candidates['class'] == 1].copy()
    non_nodules = candidates[candidates['class'] == 0].copy()
    max_neg = min(len(non_nodules), len(nodules) * 2)
    non_nodules = non_nodules.sample(n=max_neg, random_state=SEED)
    df_labels = pd.concat([nodules, non_nodules], ignore_index=True)
    df_labels = df_labels.rename(columns={'class': 'label'})
elif len(annotations) > 0:
    df_labels = annotations.copy()
    df_labels['label'] = 1
    print("AVISO: no hay candidates*.csv; usando solo annotations (todos label=1).")
else:
    raise FileNotFoundError("No se encontraron CSV dentro del dataset descomprimido.")

df_labels['mhd_path'] = df_labels['seriesuid'].astype(str).map(uid_to_mhd)
df_labels = df_labels.dropna(subset=['mhd_path'])

df_labels['group_id'] = df_labels['seriesuid'].astype(str)

print(f"\nMuestras listas para entrenamiento: {len(df_labels)}")
print(f"Distribución de label:\n{df_labels['label'].value_counts()}")
print(f"Series únicas (group_id): {df_labels['group_id'].nunique()}")


In [ ]:
# Fase 3: Dataset — carga volumen, resample, HU clip, min-max, ROI 64³
ROI_SIZE = (64, 64, 64)
HU_MIN, HU_MAX = -1000.0, 400.0

class LUNA16NoduleDataset(Dataset):
    def __init__(self, df, augment=False):
        self.records = df.reset_index(drop=True)
        self.augment = augment

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        row = self.records.iloc[idx]
        mhd_path = row['mhd_path']
        label = int(row['label'])

        try:
            img = sitk.ReadImage(mhd_path)
            # Resample a 1mm isotrópico
            orig_spacing = np.array(img.GetSpacing())
            orig_size = np.array(img.GetSize())
            new_spacing = np.array([1.0, 1.0, 1.0])
            new_size = (orig_size * orig_spacing / new_spacing).astype(int).tolist()

            resampler = sitk.ResampleImageFilter()
            resampler.SetOutputSpacing(new_spacing.tolist())
            resampler.SetSize(new_size)
            resampler.SetInterpolator(sitk.sitkLinear)
            resampler.SetOutputOrigin(img.GetOrigin())
            resampler.SetOutputDirection(img.GetDirection())
            img = resampler.Execute(img)

            vol = sitk.GetArrayFromImage(img).astype(np.float32)  # [D,H,W]

            # HU clip + min-max
            vol = np.clip(vol, HU_MIN, HU_MAX)
            vol = (vol - HU_MIN) / (HU_MAX - HU_MIN)

            # Centroide en coordenadas mundo -> voxel
            world_coord = np.array([row['coordX'], row['coordY'], row['coordZ']])
            voxel_coord = img.TransformPhysicalPointToIndex(world_coord.tolist())
            cz, cy, cx = voxel_coord[2], voxel_coord[1], voxel_coord[0]

            # Extraer ROI 64³ centrado
            half = 32
            d, h, w = vol.shape
            z0 = max(0, cz - half); z1 = z0 + 64
            y0 = max(0, cy - half); y1 = y0 + 64
            x0 = max(0, cx - half); x1 = x0 + 64

            if z1 > d: z0 = max(0, d - 64); z1 = d
            if y1 > h: y0 = max(0, h - 64); y1 = h
            if x1 > w: x0 = max(0, w - 64); x1 = w

            roi = vol[z0:z1, y0:y1, x0:x1]

            # Pad si es necesario
            pad_d = 64 - roi.shape[0]
            pad_h = 64 - roi.shape[1]
            pad_w = 64 - roi.shape[2]
            if pad_d > 0 or pad_h > 0 or pad_w > 0:
                roi = np.pad(roi, ((0, pad_d), (0, pad_h), (0, pad_w)), mode='constant')

            tensor = torch.from_numpy(roi).unsqueeze(0)  # [1, 64, 64, 64]

            # Augmentación simple
            if self.augment:
                if random.random() > 0.5:
                    tensor = torch.flip(tensor, dims=[1])
                if random.random() > 0.5:
                    tensor = torch.flip(tensor, dims=[2])
                if random.random() > 0.5:
                    tensor = torch.flip(tensor, dims=[3])

        except Exception as e:
            print(f"Error cargando {mhd_path}: {e}")
            tensor = torch.zeros(1, 64, 64, 64)

        return tensor, label

# Test rápido
_test_ds = LUNA16NoduleDataset(df_labels.head(2))
_t, _l = _test_ds[0]
print(f"Test shape: {_t.shape}, label: {_l}, min={_t.min():.3f}, max={_t.max():.3f}")


In [ ]:
# Fase 4: Modelo — Swin-Tiny con patch_embed inflado a 3D + plan B (R3D-18)

USE_SWIN_3D = True  # False para fallback R3D-18

class SwinTiny3D(nn.Module):
    """Swin-Tiny con patch_embed Conv2d → Conv3d inflado.
    Después de patch_embed_3d los 16×16×16 tokens se reorganizan en una rejilla
    2D de 64×64 (4×4 mosaico de planos de profundidad) para que los bloques Swin
    puedan aplicar ventana deslizante con sus máscaras de atención pre-calculadas.
    """
    def __init__(self, num_classes=2, pretrained=True):
        super().__init__()
        # img_size=256 → patch_size=4 → rejilla 64×64 tokens internamente
        swin2d = timm.create_model('swin_tiny_patch4_window7_224',
                                    pretrained=pretrained, num_classes=0,
                                    img_size=256)

        old_proj = swin2d.patch_embed.proj  # Conv2d(3, 96, 4, 4)
        C_out = old_proj.out_channels

        self.patch_embed_3d = nn.Conv3d(
            in_channels=1, out_channels=C_out,
            kernel_size=(4, 4, 4), stride=(4, 4, 4), padding=0,
        )
        # Init: promediar RGB→1 canal, replicar en eje Z y dividir para conservar escala
        w2d = old_proj.weight.data                       # [C_out, 3, 4, 4]
        w_mean = w2d.mean(dim=1, keepdim=True)           # [C_out, 1, 4, 4]
        w3d = w_mean.unsqueeze(2).expand(-1, -1, 4, -1, -1) / 4.0
        self.patch_embed_3d.weight.data.copy_(w3d)
        if old_proj.bias is not None:
            self.patch_embed_3d.bias.data.copy_(old_proj.bias.data)

        self.patch_embed_norm = swin2d.patch_embed.norm
        self.layers = swin2d.layers
        self.norm = swin2d.norm
        self.head = nn.Linear(swin2d.num_features, num_classes)
        self.num_features = swin2d.num_features
        del swin2d

    def forward(self, x):
        # x: [B, 1, 64, 64, 64]
        x = self.patch_embed_3d(x)              # [B, 96, 16, 16, 16]
        B, C, D, H, W = x.shape
        # Reorganizar 16 planos de profundidad en mosaico 4×4 → rejilla 64×64
        x = x.view(B, C, 4, 4, H, W)           # [B, C, 4, 4, 16, 16]
        x = x.permute(0, 1, 2, 4, 3, 5)        # [B, C, 4, 16, 4, 16]
        x = x.reshape(B, C, 4*H, 4*W)          # [B, 96, 64, 64]
        x = x.permute(0, 2, 3, 1)              # [B, 64, 64, 96] — formato Swin
        x = self.patch_embed_norm(x)

        for layer in self.layers:
            x = grad_checkpoint(layer, x, use_reentrant=False)

        x = self.norm(x)
        x = x.mean(dim=[1, 2])  # Global average pooling sobre H, W
        return self.head(x)

# Plan B: R3D-18 (si Swin3D falla en shapes/VRAM)
class R3D18Expert(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        from torchvision.models.video import r3d_18, R3D_18_Weights
        base = r3d_18(weights=R3D_18_Weights.DEFAULT)
        old_conv = base.stem[0]
        self.stem_conv = nn.Conv3d(1, 64, kernel_size=(3,7,7),
                                   stride=(1,2,2), padding=(1,3,3), bias=False)
        w = old_conv.weight.data.mean(dim=1, keepdim=True)
        self.stem_conv.weight.data.copy_(w)
        base.stem[0] = self.stem_conv
        self.backbone = nn.Sequential(*list(base.children())[:-1])
        self.head = nn.Linear(512, num_classes)

    def forward(self, x):
        feat = self.backbone(x).flatten(1)
        return self.head(feat)

# Intentar Swin3D primero
try:
    if USE_SWIN_3D:
        model = SwinTiny3D(num_classes=2).to(DEVICE)
        dummy = torch.randn(1, 1, 64, 64, 64).to(DEVICE)
        with torch.no_grad():
            out = model(dummy)
        print(f"SwinTiny3D OK — output shape: {out.shape}")
        del dummy, out
    else:
        raise RuntimeError("Usando plan B por configuración")
except Exception as e:
    print(f"SwinTiny3D falló ({e}), usando R3D-18 (plan B)")
    model = R3D18Expert(num_classes=2).to(DEVICE)
    dummy = torch.randn(1, 1, 64, 64, 64).to(DEVICE)
    with torch.no_grad():
        out = model(dummy)
    print(f"R3D18Expert OK — output shape: {out.shape}")
    del dummy, out

n_params = sum(p.numel() for p in model.parameters())
n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Params: {n_params:,} total | {n_train:,} entrenables")
torch.cuda.empty_cache()


In [ ]:
# Fase 5: Entrenamiento con StratifiedGroupKFold + AMP + checkpointing

BATCH_SIZE = 4
EPOCHS = 50
LR = 1e-4
WD = 1e-2
N_FOLDS = 3  # Usar 10 para reproducir paper; 3 para debug/tiempo
AMP_ENABLED = torch.cuda.is_available()

criterion = nn.CrossEntropyLoss()
scaler = GradScaler(enabled=AMP_ENABLED)

sgkf = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
labels_arr = df_labels['label'].values
groups_arr = df_labels['group_id'].values

all_fold_metrics = []

for fold_idx, (train_idx, val_idx) in enumerate(sgkf.split(
        np.zeros(len(df_labels)), labels_arr, groups_arr)):

    print(f"\n{'='*60}")
    print(f"FOLD {fold_idx+1}/{N_FOLDS}")
    print(f"{'='*60}")

    train_df = df_labels.iloc[train_idx]
    val_df   = df_labels.iloc[val_idx]
    print(f"Train: {len(train_df)} (pos={train_df['label'].sum()}) | "
          f"Val: {len(val_df)} (pos={val_df['label'].sum()})")

    train_ds = LUNA16NoduleDataset(train_df, augment=True)
    val_ds   = LUNA16NoduleDataset(val_df,   augment=False)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
    val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

    # Re-init model cada fold
    try:
        if USE_SWIN_3D:
            model = SwinTiny3D(num_classes=2, pretrained=True).to(DEVICE)
        else:
            raise RuntimeError()
    except:
        model = R3D18Expert(num_classes=2).to(DEVICE)

    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

    best_val_f1 = 0.0
    history = {'train_loss': [], 'val_loss': [], 'val_acc': [], 'val_f1': []}

    for epoch in range(EPOCHS):
        # ── Train ──
        model.train()
        running_loss = 0.0
        pbar = tqdm(train_loader, desc=f"F{fold_idx+1} Ep{epoch+1}", leave=False)
        for x, y in pbar:
            x = x.to(DEVICE); y = torch.tensor(y, dtype=torch.long).to(DEVICE)
            optimizer.zero_grad()
            with autocast(enabled=AMP_ENABLED):
                logits = model(x)
                loss = criterion(logits, y)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            running_loss += loss.item() * x.size(0)
            pbar.set_postfix(loss=f"{loss.item():.4f}")

        scheduler.step()
        train_loss = running_loss / len(train_ds)

        # ── Val ──
        model.eval()
        val_loss_sum = 0.0
        all_preds, all_labels = [], []
        with torch.no_grad():
            for x, y in val_loader:
                x = x.to(DEVICE); y = torch.tensor(y, dtype=torch.long).to(DEVICE)
                with autocast(enabled=AMP_ENABLED):
                    logits = model(x)
                    loss = criterion(logits, y)
                val_loss_sum += loss.item() * x.size(0)
                all_preds.extend(logits.argmax(1).cpu().numpy())
                all_labels.extend(y.cpu().numpy())

        val_loss = val_loss_sum / len(val_ds)
        val_acc = accuracy_score(all_labels, all_preds)
        val_f1  = f1_score(all_labels, all_preds, average='macro', zero_division=0)

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        history['val_f1'].append(val_f1)

        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f"  [Ep {epoch+1:3d}] TrL={train_loss:.4f} VlL={val_loss:.4f} "
                  f"Acc={val_acc:.4f} F1={val_f1:.4f}")

        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            ckpt_path = os.path.join(WEIGHTS_DIR, 'checkpoints', f'fold_{fold_idx}_best.pth')
            torch.save(model.state_dict(), ckpt_path)

    # Fin de fold
    fold_metrics = {
        'fold': fold_idx,
        'best_val_f1': best_val_f1,
        'final_val_acc': history['val_acc'][-1],
        'final_val_f1': history['val_f1'][-1],
    }
    all_fold_metrics.append(fold_metrics)
    print(f"Fold {fold_idx+1} finalizado — mejor F1: {best_val_f1:.4f}")

    # Guardar historial
    hist_path = os.path.join(WEIGHTS_DIR, 'metrics', f'history_fold_{fold_idx}.json')
    with open(hist_path, 'w') as f:
        json.dump(history, f, indent=2)

# Resumen
print(f"\n{'='*60}")
print("RESUMEN CROSS-VALIDATION")
print(f"{'='*60}")
f1s = [m['best_val_f1'] for m in all_fold_metrics]
print(f"F1 macro por fold: {[f'{v:.4f}' for v in f1s]}")
print(f"Media ± std: {np.mean(f1s):.4f} ± {np.std(f1s):.4f}")

summary_path = os.path.join(WEIGHTS_DIR, 'metrics', 'cv_summary.json')
with open(summary_path, 'w') as f:
    json.dump(all_fold_metrics, f, indent=2)
print(f"Resumen guardado en {summary_path}")


In [ ]:
# Fase 6: Evaluación detallada del mejor fold
best_fold = max(all_fold_metrics, key=lambda m: m['best_val_f1'])
best_fold_idx = best_fold['fold']
print(f"Mejor fold: {best_fold_idx} (F1={best_fold['best_val_f1']:.4f})")

# Cargar modelo del mejor fold
ckpt = os.path.join(WEIGHTS_DIR, 'checkpoints', f'fold_{best_fold_idx}_best.pth')
try:
    if USE_SWIN_3D:
        eval_model = SwinTiny3D(num_classes=2, pretrained=False).to(DEVICE)
    else:
        raise RuntimeError()
except:
    eval_model = R3D18Expert(num_classes=2).to(DEVICE)

eval_model.load_state_dict(torch.load(ckpt, map_location=DEVICE))
eval_model.eval()

# Reconstruir val del mejor fold
_, val_idx_best = list(sgkf.split(np.zeros(len(df_labels)), labels_arr, groups_arr))[best_fold_idx]
val_df_best = df_labels.iloc[val_idx_best]
val_ds_best = LUNA16NoduleDataset(val_df_best, augment=False)
val_loader_best = DataLoader(val_ds_best, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

all_preds, all_labels, all_probs = [], [], []
with torch.no_grad():
    for x, y in tqdm(val_loader_best, desc="Evaluación"):
        x = x.to(DEVICE)
        logits = eval_model(x)
        probs = F.softmax(logits, dim=1)
        all_preds.extend(logits.argmax(1).cpu().numpy())
        all_labels.extend(y.numpy() if isinstance(y, np.ndarray) else [yi for yi in y])
        all_probs.extend(probs[:, 1].cpu().numpy())

all_labels = np.array(all_labels)
all_preds = np.array(all_preds)

print("\n" + classification_report(all_labels, all_preds, target_names=["Benigno", "Maligno"]))

cm = confusion_matrix(all_labels, all_preds)
print("Confusion Matrix:")
print(cm)

# AUC si hay ambas clases
if len(np.unique(all_labels)) == 2:
    auc = roc_auc_score(all_labels, all_probs)
    print(f"AUC-ROC: {auc:.4f}")

# Guardar
eval_metrics = {
    'accuracy': float(accuracy_score(all_labels, all_preds)),
    'precision_macro': float(precision_score(all_labels, all_preds, average='macro', zero_division=0)),
    'recall_macro': float(recall_score(all_labels, all_preds, average='macro', zero_division=0)),
    'f1_macro': float(f1_score(all_labels, all_preds, average='macro', zero_division=0)),
    'confusion_matrix': cm.tolist(),
}
with open(os.path.join(WEIGHTS_DIR, 'metrics', 'eval_best_fold.json'), 'w') as f:
    json.dump(eval_metrics, f, indent=2)


In [ ]:
# Fase 7: Grad-CAM 3D
import matplotlib
matplotlib.use('Agg')
from scipy.ndimage import zoom as ndzoom

class GradCAM3D:
    """Grad-CAM genérico para tensores (B,H,W,C) [Swin] o (B,C,D,H,W) [R3D]."""
    def __init__(self, model, target_layer):
        self.model = model
        self.activations = None
        self.gradients = None
        target_layer.register_forward_hook(self._fwd_hook)
        target_layer.register_full_backward_hook(self._bwd_hook)

    def _fwd_hook(self, module, inp, out):
        self.activations = out.detach()

    def _bwd_hook(self, module, grad_in, grad_out):
        self.gradients = grad_out[0].detach()

    def __call__(self, x, class_idx=None):
        self.model.eval()
        x = x.clone().requires_grad_(True)
        logits = self.model(x)
        if class_idx is None:
            class_idx = logits.argmax(dim=1).item()
        self.model.zero_grad()
        one_hot = torch.zeros_like(logits)
        one_hot[0, class_idx] = 1.0
        logits.backward(gradient=one_hot)

        act = self.activations   # Swin norm: (B, H, W, C) | R3D: (B, C, D, H, W)
        grad = self.gradients

        if act.ndim == 4:
            # (B, H, W, C) → pesos por canal, luego suma
            weights = grad.mean(dim=[1, 2], keepdim=True)  # (B, 1, 1, C)
            cam = (weights * act).sum(dim=-1)               # (B, H, W)
        elif act.ndim == 5:
            weights = grad.mean(dim=[2, 3, 4], keepdim=True)
            cam = (weights * act).sum(dim=1)                # (B, D, H, W)
        else:
            weights = grad.mean(dim=1, keepdim=True)
            cam = (weights * act).sum(dim=-1)

        cam = F.relu(cam)
        cam = cam - cam.min()
        cam = cam / (cam.max() + 1e-8)
        return cam[0].cpu().numpy(), class_idx

# Capa objetivo: norm (último LayerNorm antes de pooling)
if hasattr(eval_model, 'norm'):
    target = eval_model.norm
elif hasattr(eval_model, 'backbone'):
    children = list(eval_model.backbone.children())
    target = children[-2] if len(children) > 1 else children[-1]
else:
    target = list(eval_model.modules())[-2]

gradcam = GradCAM3D(eval_model, target)

n_samples = min(3, len(val_ds_best))
fig, axes = plt.subplots(n_samples, 3, figsize=(15, 5*n_samples))
if n_samples == 1:
    axes = axes.reshape(1, -1)

for i in range(n_samples):
    x_sample, y_sample = val_ds_best[i]
    x_gpu = x_sample.unsqueeze(0).to(DEVICE)
    try:
        cam_raw, pred_cls = gradcam(x_gpu)
        # cam_raw puede ser 2D (H_swin, W_swin) o 3D (D,H,W)
        # Para Swin: (8, 8) mosaico → deshacer a 3D: 8=2*4 → (2,4, 2,4) → (2*2, 4*4) no aplica
        # Mejor: interpolar directamente al volumen 64³
        if cam_raw.ndim == 2:
            cam_vol = ndzoom(cam_raw, np.array([64, 64]) / np.array(cam_raw.shape), order=1)
            # Replicar en eje Z para tener un "pseudo-3D" heatmap
            cam_vol = np.stack([cam_vol]*64, axis=0)  # (64, 64, 64)
        elif cam_raw.ndim == 3:
            factors = np.array([64, 64, 64]) / np.array(cam_raw.shape)
            cam_vol = ndzoom(cam_raw, factors, order=1)
        else:
            cam_vol = np.zeros((64, 64, 64))
    except Exception as e:
        print(f"Grad-CAM error sample {i}: {e}")
        cam_vol = np.zeros((64, 64, 64))
        pred_cls = -1

    vol_np = x_sample.squeeze().numpy()
    mid = [s // 2 for s in vol_np.shape]

    slices_info = [
        ('Axial',   vol_np[mid[0]], cam_vol[mid[0]]),
        ('Coronal', vol_np[:, mid[1]], cam_vol[:, mid[1]]),
        ('Sagital', vol_np[:, :, mid[2]], cam_vol[:, :, mid[2]]),
    ]
    for j, (name, sl, cam_sl) in enumerate(slices_info):
        axes[i, j].imshow(sl, cmap='gray')
        axes[i, j].imshow(cam_sl, cmap='jet', alpha=0.4)
        axes[i, j].set_title(f"{name} | GT={y_sample} Pred={pred_cls}")
        axes[i, j].axis('off')

plt.tight_layout()
gradcam_path = os.path.join(WEIGHTS_DIR, 'figures', 'gradcam_samples.png')
plt.savefig(gradcam_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"Grad-CAM guardado en {gradcam_path}")


In [ ]:
# Fase 8: Figuras de entrenamiento (curvas por fold)
fig_dir = os.path.join(WEIGHTS_DIR, 'figures')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for fold_idx in range(N_FOLDS):
    hp = os.path.join(WEIGHTS_DIR, 'metrics', f'history_fold_{fold_idx}.json')
    if not os.path.isfile(hp):
        continue
    with open(hp) as f:
        h = json.load(f)
    axes[0].plot(h['train_loss'], label=f'Train F{fold_idx}', alpha=0.7)
    axes[0].plot(h['val_loss'], '--', label=f'Val F{fold_idx}', alpha=0.7)
    axes[1].plot(h['val_f1'], label=f'F1 F{fold_idx}', alpha=0.7)

axes[0].set_title('Loss'); axes[0].set_xlabel('Epoch'); axes[0].legend()
axes[1].set_title('Val F1 Macro'); axes[1].set_xlabel('Epoch'); axes[1].legend()
axes[1].axhline(y=0.65, color='red', linestyle='--', label='Consigna (0.65)')
axes[1].legend()

plt.tight_layout()
curves_path = os.path.join(fig_dir, 'training_curves.png')
plt.savefig(curves_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"Curvas guardadas en {curves_path}")

# Confusion matrix heatmap
import seaborn as sns
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Benigno', 'Maligno'],
            yticklabels=['Benigno', 'Maligno'])
plt.title('Confusion Matrix — Expert 4 LUNA16')
plt.xlabel('Predicted'); plt.ylabel('True')
cm_path = os.path.join(fig_dir, 'confusion_matrix.png')
plt.savefig(cm_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"CM guardada en {cm_path}")


In [ ]:
# Fase 9: Resumen final y persistencia
print("="*60)
print("EXPERTO 4 — LUNA16 SWIN3D — RESUMEN")
print("="*60)

print(f"\nModelo: {'SwinTiny3D' if USE_SWIN_3D else 'R3D-18'}")
print(f"Parámetros: {n_params:,}")
print(f"Folds: {N_FOLDS}")
print(f"Épocas/fold: {EPOCHS}")
print(f"Batch size: {BATCH_SIZE}")
print(f"LR: {LR}, WD: {WD}")
print(f"AMP: {AMP_ENABLED}")
print(f"Device: {DEVICE}")

print(f"\nMétricas CV:")
for m in all_fold_metrics:
    print(f"  Fold {m['fold']}: F1={m['best_val_f1']:.4f}")

f1s = [m['best_val_f1'] for m in all_fold_metrics]
print(f"  Media: {np.mean(f1s):.4f} ± {np.std(f1s):.4f}")

print(f"\nArtefactos en Drive:")
print(f"  {WEIGHTS_DIR}")
for root, dirs, files in os.walk(WEIGHTS_DIR):
    for f_name in files:
        fp = os.path.join(root, f_name)
        sz = os.path.getsize(fp) / 1024
        print(f"    {os.path.relpath(fp, WEIGHTS_DIR):50s} {sz:.1f} KB")

print("\nExperto 4 listo para integración MoE.")
print("Tensor de entrada esperado: [B, 1, 64, 64, 64]")
print("Salida: logits [B, 2] (benigno/maligno)")
